# Morphy scenario notebook

This notebook runs the predefined viability-kernel scenarios using the current repository structure, summarizes the viable fraction for each scenario, and saves a multi-panel phase-portrait figure to `../figures/all_scenarios_from_notebook.png`. It is written to stay consistent with the repo as currently named: top-level `config/` plus the scientific package `viability_kernels`.[file:71][file:74][file:75][file:76][file:79]

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FIG_PATH = FIGURES_DIR / 'all_scenarios_from_notebook.png'

print(f'Repo root: {ROOT}')
print(f'Figure will be saved to: {FIG_PATH}')


In [ ]:
from config import DEFAULT_PARAMS, DEFAULT_BOUNDS, DEFAULT_SIM, SCENARIOS
from viability_kernels.simulation import run_scenario
from viability_kernels.phase_plane import plot_all_scenarios

print(f'Loaded {len(SCENARIOS)} scenarios.')


In [ ]:
x0_center = np.array(DEFAULT_SIM['x0_center'])
results = []

for cfg in SCENARIOS:
    result = run_scenario(
        scenario_cfg=cfg,
        par=DEFAULT_PARAMS,
        bounds=DEFAULT_BOUNDS,
        x0_center=x0_center,
        n_traj=DEFAULT_SIM['n_traj'],
        t_span=tuple(DEFAULT_SIM['t_span']),
        n_eval=DEFAULT_SIM['n_eval'],
        rng_seed=DEFAULT_SIM['rng_seed'],
    )
    result['expected'] = cfg.get('expected', '—')
    result['description'] = cfg.get('description', '')
    results.append(result)

summary = pd.DataFrame([
    {
        'label': r['label'],
        'p': r['p'],
        'viable_fraction': r['viable_fraction'],
        'viable_percent': round(100 * r['viable_fraction'], 1),
        'expected': r['expected'],
    }
    for r in results
])
summary = summary.sort_values(['p', 'label']).reset_index(drop=True)
summary


In [ ]:
fig = plot_all_scenarios(
    scenario_results=results,
    par=DEFAULT_PARAMS,
    bounds=DEFAULT_BOUNDS,
    suptitle='Viability kernel simulations — notebook export',
    save_path=str(FIG_PATH),
)
plt.show()

print(f'Saved figure: {FIG_PATH}')


In [ ]:
assert FIG_PATH.exists(), f'Expected figure not found at {FIG_PATH}'
FIG_PATH
